<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/sistema_graphrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema GraphRAG — Leyes de inversión de Paraguay

Chatbot legal sobre derecho de inversiones, con **conciencia de vigencia**: sabe qué normas están derogadas y avisa.

## Cómo está organizado

| Sección | Qué hace | ¿Cuándo correrla? |
|---|---|---|
| **0. Setup** | Dependencias, credenciales, config, conexión | **Siempre**, al abrir el notebook |
| **1. Carga** | Embeddings de artículos + fichas | **Una sola vez** (es idempotente: re-correrla no rompe nada) |
| **2. Retriever** | Recuperación GraphRAG + pipeline con LLM | **Siempre** |
| **3. Baseline** | El mismo sistema *sin* grafo, para comparar | Para la evaluación |
| **4. Agente** | Chat multi-turno con memoria y auto-corrección | Para conversar |

**Uso normal:** corré la Sección 0 y después la 2 (o directamente `Runtime → Run all`).
La Sección 1 ya está hecha en la base; si la corrés de nuevo dirá "pendientes: 0" y sigue.

## Antes de empezar
En **Colab → Secrets** (🔑 en la barra izquierda) cargá, con acceso a este notebook:
`OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`

---
# 0. Setup

Correr siempre. Deja listos: `driver`, `embedder`, `llm`, `prompt_template` y toda la config.

In [1]:
# Todas las dependencias en UNA sola instalación (evita choques de versión).
!pip install -q "neo4j_graphrag[openai]" langgraph tiktoken tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 9.0 MB/s eta 0:00:00


In [2]:
# Credenciales. En Colab: cargalas en Secrets (icono de la llave) con estos nombres.
import os
_KEYS = ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]
try:
    from google.colab import userdata
    for k in _KEYS:
        os.environ[k] = userdata.get(k)
    print("Credenciales cargadas desde Colab Secrets.")
except Exception:
    import getpass
    for k in _KEYS:
        if not os.environ.get(k):
            os.environ[k] = getpass.getpass(f"{k}: ")
    print("Credenciales cargadas por teclado.")

Credenciales cargadas desde Colab Secrets.


### Configuración

Acá vive todo lo ajustable: modelos, nombres del esquema, índices y `TOP_K`.

In [3]:
import os, time
from openai import OpenAI
from neo4j import GraphDatabase
import tiktoken
from tenacity import retry, wait_random_exponential, stop_after_attempt

# Silenciar el warning de deprecación de db.index.vector.queryNodes (inofensivo):
import logging; logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

# --- Conexión ---
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USER     = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]     # OJO: la base es 'ece63d51', no 'neo4j'

# --- Modelos ---
EMBEDDING_MODEL      = "text-embedding-3-small"   # el MISMO para indexar y para consultar
EMBEDDING_DIM        = 1536
EMBEDDING_ENCODING   = "cl100k_base"
MAX_TOKENS_PER_INPUT = 8000
LLM_MODEL            = "gpt-4o"

# --- Esquema del grafo ---
LABEL_ARTICULO    = "Articulo"
LABEL_FICHA       = "Ficha"
LABEL_RECUPERABLE = "Recuperable"      # etiqueta compartida: es la que indexa el vector
REL_TIENE_ART     = "TIENE_ARTICULO"
NORM_LABELS       = ["Ley", "Decreto", "Resolucion"]
PROP_TEXTO        = "texto"
PROP_NUM_ART      = "numero"
PROP_PARTE        = "parte"
PROP_LEY_NUMERO   = "ley_numero"
PROP_NUM_NORMA    = "numero"
PROP_NOMBRE_COMPLETO = "nombre_completo"
PROP_EMBEDDING    = "embedding"
PROP_ES_PRUEBA    = "es_prueba"

# --- Índices ---
VECTOR_INDEX_NAME   = "recuperable_embedding"
FULLTEXT_ART_NAME   = "articulo_texto_ft"
FULLTEXT_NORMA_NAME = "normas_nombre_ft"

# --- Ajustables ---
EMBED_BATCH_SIZE = 50
WRITE_BATCH_SIZE = 500
TEST_LIMIT       = None    # None = todo el corpus. Poné 20 para una prueba chica.
TOP_K            = 5

# --- Clientes compartidos ---
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
enc    = tiktoken.get_encoding(EMBEDDING_ENCODING)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado. Config lista.")

Conectado. Config lista.


### Objetos compartidos

Se definen acá (no dentro de un pipeline) para que las secciones 2, 3 y 4 no dependan del orden de las celdas.

In [4]:
# Embedder, LLM y prompt COMPARTIDOS: los usan el retriever, el baseline y el agente.
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import RagTemplate

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)
llm = OpenAILLM(model_name=LLM_MODEL, model_params={"temperature": 0})

TEMPLATE = """Sos un asistente legal sobre derecho de inversiones de Paraguay.
Respondé usando SOLO el contexto. Citá la norma y el artículo en cada afirmación.
Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo
explícitamente y NO la presentes como vigente. Si una FICHA advierte que hace
referencia a normas no vigentes, trasladá esa advertencia al usuario.
Si el contexto no alcanza, decilo.

# Contexto:
{context}
{examples}
# Pregunta:
{query_text}

# Respuesta:"""
prompt_template = RagTemplate(template=TEMPLATE)
print("Embedder, LLM y prompt listos.")

Embedder, LLM y prompt listos.


---
# 1. Carga (correr una sola vez)

Genera los **embeddings** y crea los **índices**. Todo va al mismo índice vectorial gracias a la etiqueta compartida `:Recuperable`.

**Es idempotente:** saltea lo que ya está hecho. Si ya lo corriste, no vuelve a pagar ni a recalcular.

### Funciones de carga

In [5]:
# Funciones de carga. Se usan en 1.1 (artículos) y 1.2 (fichas).

def crear_indices(driver):
    # Índice vectorial sobre la etiqueta COMPARTIDA :Recuperable, no sobre :Articulo.
    # Así los Artículos Y las Fichas entran al MISMO índice sin recrearlo.
    driver.execute_query(
        f"""
        CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
        FOR (n:{LABEL_RECUPERABLE}) ON (n.{PROP_EMBEDDING})
        OPTIONS {{ indexConfig: {{
            `vector.dimensions`: {EMBEDDING_DIM},
            `vector.similarity_function`: 'cosine'
        }} }}
        """,
        database_=NEO4J_DATABASE,
    )
    # Fulltext sobre texto libre. Los números con "/" NO van acá (el tokenizer los parte):
    # esos se buscan con match exacto -> MATCH (n) WHERE n.numero = '60/90'
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_ART_NAME} IF NOT EXISTS "
        f"FOR (a:{LABEL_ARTICULO}) ON EACH [a.{PROP_TEXTO}]",
        database_=NEO4J_DATABASE,
    )
    normas = "|".join(NORM_LABELS)
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_NORMA_NAME} IF NOT EXISTS "
        f"FOR (n:{normas}) ON EACH [n.{PROP_NOMBRE_COMPLETO}]",
        database_=NEO4J_DATABASE,
    )
    print("Índices creados/verificados.")


def recortar_a_limite(texto):
    toks = enc.encode(texto)
    if len(toks) <= MAX_TOKENS_PER_INPUT:
        return texto, len(toks), False
    return enc.decode(toks[:MAX_TOKENS_PER_INPUT]), MAX_TOKENS_PER_INPUT, True


@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(6))
def embeber_lote(textos):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=textos)
    return [d.embedding for d in resp.data]


def escribir_vectores(driver, filas):
    # Escribe el vector Y agrega la etiqueta :Recuperable (para que el índice lo tome).
    query = f"""
        UNWIND $rows AS row
        MATCH (a) WHERE elementId(a) = row.eid
        CALL db.create.setNodeVectorProperty(a, $prop, row.embedding)
        SET a:{LABEL_RECUPERABLE}
    """
    for i in range(0, len(filas), WRITE_BATCH_SIZE):
        lote = filas[i:i + WRITE_BATCH_SIZE]
        driver.execute_query(query, rows=lote, prop=PROP_EMBEDDING, database_=NEO4J_DATABASE)


def leer_articulos_pendientes(driver):
    norm_pred = " OR ".join(f"n:{l}" for l in NORM_LABELS)
    limit_clause = f"LIMIT {TEST_LIMIT}" if TEST_LIMIT else ""
    query = f"""
        MATCH (a:{LABEL_ARTICULO})
        WHERE a.{PROP_TEXTO} IS NOT NULL AND trim(a.{PROP_TEXTO}) <> ''
              AND a.{PROP_EMBEDDING} IS NULL
              AND coalesce(a.{PROP_ES_PRUEBA}, false) = false
        OPTIONAL MATCH (n)-[:{REL_TIENE_ART}]->(a)
            WHERE {norm_pred}
        WITH a, collect(n)[0] AS norm
        RETURN elementId(a) AS eid,
               a.{PROP_NUM_ART}    AS art_num,
               a.{PROP_PARTE}      AS parte,
               a.{PROP_TEXTO}      AS texto,
               a.{PROP_LEY_NUMERO} AS ley_numero,
               CASE WHEN norm IS NULL THEN null
                    ELSE head([l IN labels(norm) WHERE l IN {NORM_LABELS}]) END AS norm_tipo,
               norm.{PROP_NUM_NORMA} AS norm_num
        {limit_clause}
    """
    records, _, _ = driver.execute_query(query, database_=NEO4J_DATABASE)
    return [r.data() for r in records]


def texto_con_encabezado(row):
    # Embebemos "Ley 60/90, Artículo 5: <texto>" y no el texto pelado, para que el
    # vector sepa a qué norma pertenece.
    partes = []
    tipo = row.get("norm_tipo")
    num = row.get("norm_num") or row.get("ley_numero")
    if tipo and num:
        partes.append(f'{tipo} {num}')
    elif num:
        partes.append(f'{num}')
    if row.get("art_num") is not None:
        partes.append(f'Artículo {row["art_num"]}')
    if row.get("parte") and row["parte"] not in (None, "", "cuerpo"):
        partes.append(f'({row["parte"]})')
    prefijo = ", ".join(partes)
    return f'{prefijo}: {row["texto"]}' if prefijo else row["texto"]


def embeber_y_guardar(driver, preparados, etiqueta="ítems"):
    # preparados = [(elementId, texto_final), ...]
    total_tokens = sum(len(enc.encode(t)) for _, t in preparados)
    print(f"  Tokens aprox: {total_tokens:,} | costo estimado: ~US${total_tokens/1_000_000*0.02:.4f}")
    hechos = 0
    for i in range(0, len(preparados), EMBED_BATCH_SIZE):
        lote = preparados[i:i + EMBED_BATCH_SIZE]
        vectores = embeber_lote([t for _, t in lote])
        filas = [{"eid": eid, "embedding": v} for (eid, _), v in zip(lote, vectores)]
        escribir_vectores(driver, filas)
        hechos += len(filas)
        print(f"  {hechos}/{len(preparados)} {etiqueta} embebidos y guardados")
        time.sleep(0.2)

### 1.1 — Artículos

Embebe el texto de cada artículo con un encabezado de contexto (`"Ley 60/90, Artículo 5: ..."`).

In [6]:
# 1.1 — Índices + embeddings de los ARTÍCULOS.
# IDEMPOTENTE: saltea lo ya embebido. Si ya lo corriste, dirá "pendientes: 0".
crear_indices(driver)

pendientes = leer_articulos_pendientes(driver)
print(f"Artículos pendientes de embeber: {len(pendientes)}")

if pendientes:
    preparados, recortados = [], []
    for row in pendientes:
        texto_final, _, fue_recortado = recortar_a_limite(texto_con_encabezado(row))
        preparados.append((row["eid"], texto_final))
        if fue_recortado:
            recortados.append(row["eid"])
    if recortados:
        print(f"AVISO: {len(recortados)} artículos se recortaron por límite de tokens.")
    embeber_y_guardar(driver, preparados, "artículos")

# Verificación
recs, _, _ = driver.execute_query(
    f"""MATCH (a:{LABEL_ARTICULO})
        RETURN count(a) AS total,
               count(a.{PROP_EMBEDDING}) AS con_embedding,
               sum(CASE WHEN a.{PROP_TEXTO} IS NOT NULL AND a.{PROP_EMBEDDING} IS NULL
                        THEN 1 ELSE 0 END) AS pendientes""",
    database_=NEO4J_DATABASE,
)
print("Verificación artículos:", recs[0].data())
print("(Los que quedan sin embedding son stubs SIN texto: normas citadas pero no cargadas.)")

Índices creados/verificados.
Artículos pendientes de embeber: 0
Verificación artículos: {'total': 1470, 'con_embedding': 1442, 'pendientes': 0}
(Los que quedan sin embedding son stubs SIN texto: normas citadas pero no cargadas.)


### 1.2 — Fichas (documentos operativos del MIC)

Las fichas aportan lo que el articulado **no tiene**: REQUISITOS, PROCESO, plazos y costos — que es lo que realmente pregunta un inversor.

**El punto clave:** cada ficha se enlaza con `(Ficha)-[:DESCRIBE]->(norma)`, y el retriever trae la **vigencia** de esas normas. Varias fichas oficiales del MIC citan la **Ley 60/90 (derogada)** — incluso la página del PPA consultada el 11/07/2026. El grafo las corrige: **una ficha desactualizada no puede presentar como vigente un régimen que ya no lo está.**

Mirá el **PRE-CHECK** que imprime: te dice qué normas referenciadas existen en el grafo y cuáles no.

In [7]:
# =============================================================================
# PARTE A2 — Cargar FICHAS (documentos operativos MIC/REDIEX)
# =============================================================================
# Las fichas aportan lo que el articulado NO tiene: REQUISITOS, PROCESO y
# beneficios explicados. Se cargan como nodos :Ficha con etiqueta :Recuperable
# (entran al MISMO indice vectorial, sin recrear nada) y se enlazan con
# (Ficha)-[:DESCRIBE]->(norma).
#
# CLAVE: el retrieval_query expande por DESCRIBE y trae la VIGENCIA de cada
# norma descrita -> el grafo mantiene HONESTAS a las fichas. Varias fichas
# oficiales del MIC citan normas derogadas (ej. la 60/90); el sistema lo marca.
#
# Requisitos: haber corrido el Setup (driver, client, enc, EMBEDDING_MODEL).
# Es IDEMPOTENTE: se puede re-correr sin duplicar ni re-embeber.
# =============================================================================

FICHAS = [
    # -------------------------------------------------------------------------
    # HISTÓRICA: describe la Ley 60/90, DEROGADA por la 7548/25.
    # Se carga porque la 7548/25 SÍ está en el grafo (puede corregir), y porque
    # su Art. 35 mantiene las reglas viejas para proyectos ya aprobados.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_incentivos_fiscales_60_90",
        "titulo": "Régimen de Incentivos Fiscales para la Inversión de Capital (Ley 60/90)",
        "tipo": "ficha_mic",
        "estado": "historica",
        "fuente": "MIC",
        "nota": ("Describe el régimen de la Ley 60/90, DEROGADA por la Ley 7548/25. "
                 "Se conserva por valor transicional: el Art. 35 de la Ley 7548/25 mantiene "
                 "las disposiciones anteriores para los proyectos ya acogidos a la 60/90."),
        "describe": ["60/90", "22031"],
        "texto": """OBJETO: Promover e incrementar las inversiones de capital de origen nacional y/o extranjero que tengan por objeto: acrecentar la producción de bienes y servicios; crear fuentes de trabajo permanente; fomentar las exportaciones y sustituir importaciones; incorporar tecnologías que permitan aumentar la eficiencia productiva y mayor utilización de materias primas, mano de obra y recursos energéticos nacionales; y la reinversión de utilidades en bienes de capital. Beneficia a todos los sectores: Industrial, Agropecuario, Minas y Canteras, y Servicios.

BENEFICIOS QUE OFRECE:
- Arancel 0% para importación de bienes de capital (maquinarias y equipos que no son fabricados en Paraguay).
- Impuesto al Valor Agregado (IVA) 0% sobre bienes de capital (que no son fabricados en Paraguay cuando se trata de importaciones, y que son fabricados en Paraguay cuando se trata de compras locales).
- Exoneración del impuesto aplicado a las remesas y pagos en concepto de intereses para inversiones mayores a 5 millones de US$.
- Exoneración del impuesto sobre las remesas de dividendos y utilidades para inversiones mayores a 5 millones de US$ por 10 años, siempre que no provenga de un territorio de baja o nula tributación o no sea crédito fiscal en el país inversor.

REQUISITOS: Presentar por Nota solicitud ante el Ministerio de Industria y Comercio (MIC), acompañada de:
- Proyecto de inversión e Informe de Inversión en caso de reinversión.
- Constitución de sociedad.
- Acta de Asamblea.
- Antecedentes judiciales de directores y cédula de identidad del representante legal.
- Título de propiedad coincidente con la Licencia Ambiental.
- Autorización de organismos competentes (permisos habilitantes conforme al sector).
- Certificado de funcionamiento de bienes de capital (cuando la antigüedad del bien supere los 5 años de fabricación).
- Constancia de RUC (Subsecretaría de Estado de Tributación - SET).
- Certificado de cumplimiento tributario y del seguro social (SET e IPS respectivamente).
- Constancia en el Registro de Personas Jurídicas y de Beneficiarios Finales (Abogacía del Tesoro).
- Despacho de importación (en caso de importación provisoria).
- Estados financieros (3 últimos ejercicios cerrados: Balance General, Estado de Resultados, Estado de flujo de efectivo, cambios del patrimonio neto y notas).
- Facturas pro forma de bienes de capital a importar y/o compra local.
- Inscripción en el Banco Central del Paraguay (empresas con capital extranjero).
- Licencia ambiental (Ministerio del Ambiente).
- Autorización de la SET (formato DDI).
- Referencia bancaria.
- RIEL activo para empresas que están operando; las que están por iniciar operaciones deben inscribirse en un plazo no mayor a 6 meses luego de la importación de bienes de capital.
- Contrato o constancia de la entidad financiera que proveerá el crédito (si solicita el inciso "f" del Art. 5º de la Ley Nº 60/90).

PROCESO: La Dirección del Viceministerio de Industria realiza el chequeo documental y eleva el expediente al Consejo de Inversiones (órgano mixto público-privado), que dictamina favorablemente o solicita información complementaria. En caso de dictamen favorable, los beneficios son otorgados por Resolución biministerial suscrita por el Ministro de Industria y Comercio (MIC) y el Ministro de Hacienda (MH). El organismo de aplicación es el MIC, y el MH está a cargo de los aspectos tributarios.

LEGISLACIÓN RESPALDATORIA: Ley Nº 60/90; Decreto reglamentario Nº 22.031/2003, modificado por el Decreto Nº 6427/2005 y el Decreto Nº 11.462/2013.""",
    },

    # -------------------------------------------------------------------------
    # VIGENTE. Ojo: la ficha cita incentivos de la Ley 60/90 (DEROGADA) ->
    # el grafo lo marcará vía DESCRIBE.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_zonas_francas_523_95",
        "titulo": "Régimen de Zonas Francas (Ley 523/95)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("La ficha menciona la concesión de incentivos fiscales de la Ley 60/90, "
                 "que está DEROGADA por la Ley 7548/25."),
        "describe": ["523/95", "60/90"],
        "texto": """OBJETO: Promover la atracción de inversión productiva, diversificar las exportaciones, la generación de empleo y la transferencia de conocimiento y especialización de mano de obra paraguaya.

ALCANCE: Las Zonas Francas son espacios del territorio nacional, sujetas al control fiscal, aduanero y administrativo, en las cuales se pueden desarrollar actividades comerciales, industriales y de servicios. Es CONCESIONARIO la persona jurídica que, mediante contrato celebrado con el Poder Ejecutivo, adquiere el derecho de habilitar, administrar y explotar una Zona Franca, otorgado por 30 años de plazo, prorrogables. Es USUARIO la persona física o jurídica que desarrolla actividades comerciales, industriales y/o de servicios dentro de la Zona Franca.

BENEFICIOS QUE OFRECE:
- Importación de materia prima o mercaderías con una tasa de 0%.
- Impuesto al Valor Agregado (IVA) 0%.
- Servicios y comercios entre usuarios de Zonas Francas: 0% de impuestos.
- Exportación a terceros países: 0,5% del valor factura.
- Emisión de certificados de origen para los productos fabricados en Zona Franca que cumplan con los requisitos de origen MERCOSUR (ROM).
- Administración de Aduanas (DNA) instalada en la Concesionaria para agilizar los trámites de tránsito, introducción, importación y exportación.
- Disponibilidad de infraestructura inmobiliaria para todas las actividades.
- Alta disponibilidad de energía eléctrica de calidad a costo competitivo.
- Bajo costo de operación para fabricar y vender a clientes de Paraguay o países vecinos.
- Sin pérdida de origen de los productos introducidos en la zona franca (Ley Nº 523/95, Art. 20 y Decreto Nº 7068/2006).
- No requiere contratar póliza de seguro para garantías aduaneras.
- Concesión de incentivos fiscales de la Ley Nº 60/90 a la inversión nacional y extranjera.

REQUISITOS PARA SER CONCESIONARIO: El postulante deberá presentar ante el Consejo Nacional de Zonas Francas (CNZF) un proyecto de inversión que demuestre fehacientemente su viabilidad económica, conteniendo (Decreto Nº 15554/96, Art. 23):
a. Determinación de la forma o modalidad jurídica de la empresa a través de la cual se realizará la explotación.
b. La localización del predio y la superficie en que se propone desarrollar el proyecto.
c. Causas y consecuencias de su emplazamiento.
d. La posibilidad de su expansión futura.
e. Los servicios que se propone suministrar y el monto de inversión en servicios, indicando las responsabilidades de ejecución.
f. Descripción de las inversiones en infraestructura (caminos, cercado, construcciones, etc.).
g. Fuentes de financiamiento.
h. Tiempo estimado de realización del proyecto y fecha de comienzo de las obras; si se desarrolla por etapas, la superficie, obras e infraestructura de cada etapa y su tiempo de realización.
i. Estudio de mercado con indicación de cantidad y calidad de posibles Usuarios.
j. Estimación del personal a utilizar, tanto en el funcionamiento de la Zona como por parte de las empresas a instalarse.
k. Previsiones para el tratamiento de efluentes, eliminación de residuos y medidas de protección del medio ambiente.
l. Estimación del precio a cobrar a los Usuarios por el alquiler y/o venta de predios y construcciones.
m. Requerimiento de obras de infraestructura de apoyo por parte del Gobierno o empresas estatales, departamentales o municipales (caminos de acceso, puertos, tendido eléctrico, telefónico, etc.).
Los predios donde se instale la Zona Franca deberán ser propiedad del Concesionario o existir una relación contractual entre este y el propietario, por un plazo mínimo igual al fijado para la concesión (Decreto Nº 15554/96, Art. 15).

REQUISITOS PARA SER USUARIO:
- Contrato celebrado con el Concesionario.
- Inscripción en los registros nacionales de constitución de sociedad, RUC y patentes.
- Certificado de no hallarse en quiebra y de no tener inhibición de bienes.

PROCESO PARA LA CONCESIÓN DE UNA ZONA FRANCA: El postulante presenta ante el CNZF un proyecto de inversión conforme al Decreto Nº 15554/96. El CNZF estudia el proyecto y, con dictamen fundado, lo eleva al Poder Ejecutivo. De ser aprobado, se suscribe el contrato entre el Poder Ejecutivo y el postulante a Concesionario. Una vez inscripto, queda habilitado para comenzar las obras, pudiendo ingresar libre de tributos los materiales, bienes y equipos necesarios.

PROCESO PARA SER USUARIO DE UNA ZONA FRANCA: Se presenta la solicitud ante el Concesionario de la Zona Franca conforme a los requisitos establecidos. Estos se presentan a la Dirección Ejecutiva del CNZF, adjuntando el contrato respectivo y las demás documentaciones. La Dirección Ejecutiva expide la Constancia de Usuario, una vez cumplidos los requisitos, en un plazo no mayor a 48 horas hábiles.

LEGISLACIÓN RESPALDATORIA: Ley Nº 523/1995; Decreto reglamentario Nº 15554/1996; Decreto Nº 19461/2002; Decreto Nº 21309/2003; Decreto Nº 952/2018; Decreto Nº 4611/2020; Resolución General Nº 80/2021 de la SET.""",
    },

    # -------------------------------------------------------------------------
    # Ley 1064/97: derogación DIFERIDA (vigente_hasta) -> el grafo lo marcará.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_maquila_1064_97",
        "titulo": "Régimen de Maquila (Ley 1064/97)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("Verificar la vigencia de la Ley 1064/97 en el grafo: tiene derogación "
                 "diferida (vigente_hasta). La ficha no lo menciona."),
        "describe": ["1064/97", "9585"],
        "texto": """OBJETO: Promover el establecimiento y regular las operaciones de empresas industriales maquiladoras que se dediquen total o parcialmente a realizar procesos industriales o de servicios, incorporando mano de obra y otros recursos nacionales destinados a la transformación, elaboración, reparación o ensamblaje de mercaderías de procedencia extranjera importadas temporalmente a dicho efecto, para su reexportación posterior, en ejecución de un contrato suscrito con una empresa domiciliada en el extranjero.

ALCANCE: Beneficia a cualquier persona física o jurídica, nacional o extranjera, legalmente constituida en Paraguay, que se encuentre habilitada para realizar actos de comercio y que guarde relación comercial con otra empresa en el exterior. También a personas físicas o jurídicas con capacidad ociosa. Las industrias maquiladoras podrán instalarse en cualquier lugar del territorio paraguayo, adecuándose a los requisitos locales según el caso.

BENEFICIOS QUE OFRECE:
- 1% de tributo único sobre el valor agregado en territorio nacional, o sobre el valor de la factura emitida por orden y cuenta de la matriz, el que resulte mayor (Tributo Único Maquila).
- Suspensión de aranceles e impuestos a la importación de materias primas, insumos y bienes de capital.
- Exoneración de las tasas aduaneras, portuarias y aeroportuarias.
- Exoneración de tributos que gravan la remesa de dinero relacionada al régimen de maquila.
- Recuperación del crédito fiscal (IVA) correspondiente a la adquisición de bienes y servicios aplicados en forma directa o indirecta a las operaciones de maquila.
- Exoneración de impuestos departamentales o municipales (maquiladoras puras).

REQUISITOS: El trámite de inscripción al régimen de maquila es electrónico y se realiza a través de la plataforma VUE, en el módulo exclusivo para el régimen de maquila, adjuntando:
- Escritura Pública de Constitución (para empresas).
- Constancia de Inscripción en el Registro de Beneficiarios Finales y Personas Jurídicas.
- Documento de identidad de las personas físicas que solicitan su inscripción, o de los representantes de las personas jurídicas.
- Constancia de RUC.
- RUC de la empresa o persona.
- Acta actualizada de designación de directorio.

El PROGRAMA DE MAQUILA deberá contener, entre otros datos:
- Datos del solicitante.
- Características del programa de maquila, el tipo de programa a implementar y la forma de operación.
- Datos relativos a la actividad a desarrollar o servicios a prestar.
- Datos sobre la mano de obra a generar, materias primas, insumos y maquinarias a utilizar en el proceso maquilador, mercados de proveedores y de destino de los productos maquilados, exportación, importación y valor agregado nacional.
El programa deberá cargarse en la plataforma VUE acompañado de los recaudos documentales correspondientes.

PROCESO: Los interesados solicitan los beneficios del régimen a través de la plataforma VUE, pidiendo usuario y contraseña para el Régimen de Maquila; los procesos de inscripción y aprobación del programa de maquila son electrónicos. El Programa de Maquila es tratado por el Consejo Nacional de Industrias Maquiladoras de Exportación (CNIME) y, en caso de dictamen favorable, la decisión se formaliza por Resolución Biministerial (MIC-MH). Tiempo aproximado de duración del trámite: 90 días.

LEGISLACIÓN RESPALDATORIA: Ley Nº 1064/97; Decreto reglamentario Nº 9585/2000.""",
    },

    # -------------------------------------------------------------------------
    # VIGENTE. Ojo: cita requisitos y beneficios de la Ley 60/90 (DEROGADA).
    # -------------------------------------------------------------------------
    {
        "id": "ficha_garantia_inversiones_5542_15",
        "titulo": "Régimen de Garantía de Inversiones (Ley 5542/2015)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("La ficha remite a los beneficios y requisitos de la Ley 60/90, "
                 "que está DEROGADA por la Ley 7548/25."),
        "describe": ["5542/15", "60/90"],
        "texto": """OBJETO: La protección de la inversión de capital en la creación de industrias u otras actividades productivas, cuando ellas contribuyan a la generación de empleo y al desarrollo económico y social, principalmente a través de la incorporación de valor agregado a la materia prima paraguaya o importada.

ALCANCE: Beneficia a las personas físicas o jurídicas, nacionales o extranjeras, que inviertan capital para la creación de empresas o que adquieran empresas existentes y cumplan con los objetivos mencionados.

OBLIGACIONES DE LA EMPRESA:
- Incorporar la totalidad del capital en el plazo establecido en el contrato.
- Sujeción a la legislación nacional, y en particular a las disposiciones en materia ambiental y de salud pública.
- Fiel cumplimiento del contrato.
- Someter los estados financieros anuales a auditoría externa.
- Presentar una declaración sobre las inversiones efectuadas en un año y abonar un canon anual equivalente al 1% (uno por ciento) de dicha inversión.

BENEFICIOS QUE OFRECE:
- Invariabilidad de la tasa impositiva del impuesto a la renta que grava la actividad desarrollada: por un plazo de hasta 10 años para inversiones menores a 50 millones de US$; 15 años para inversiones entre 50 y 100 millones de US$; y 20 años para inversiones de 100 millones de US$ y más.
- Libre transferencia de remesas de capital (a los 2 años desde la puesta en marcha) y de utilidades líquidas sin límite de tiempo.
- Régimen arancelario correspondiente a la importación de maquinarias y equipos que no se produzcan en el país, conforme a los beneficios de la Ley Nº 60/90 (arancel 0% para bienes de capital y 0% de IVA sobre los mismos).
- Régimen especial para la exportación: podrán mantener un porcentaje de divisas en el exterior cuando sean necesarias para pagar obligaciones legalmente autorizadas o para cumplir con la remesa de utilidades.
- Invariabilidad tributaria para la compra de empresas existentes o cuando se transfiera parte de sus acciones.
- Beneficios adicionales para las industrias de alto contenido social y sus accionistas: exoneración de la tasa adicional del 5% del impuesto a la renta por la distribución de utilidades; y disminución de la tasa impositiva aplicada a la remisión de utilidades al exterior en un 1% por cada 100 empleos directos generados, hasta un máximo del 50% del valor total de la tasa aplicable.
- Seguridad jurídica: las inversiones no podrán ser objeto de ninguna modalidad de apropiación ni confiscación, y están protegidas por el principio de irretroactividad de la ley.

REQUISITOS: Presentar solicitud por Nota ante el Ministerio de Industria y Comercio, acompañada de todos los requisitos previstos para ser beneficiaria de la Ley Nº 60/90 (proyecto y cronograma de inversión) y además:
- Nota de autorización al Equipo Económico Nacional, al Consejo de Inversiones, al Ministerio de Industria y Comercio (MIC) y al Ministerio de Hacienda (MH) para requerir y obtener información y datos obrantes en instituciones públicas y privadas, nacionales y extranjeras.
- Certificado de cumplimiento tributario y Constancia de RUC (SET).
- Certificado de cumplimiento con el seguro social (IPS) o constancia de inscripción patronal.
- Últimos 3 Estados Financieros cerrados, o Balance de Apertura para empresas nuevas.
- Referencia bancaria.
- Inscripción de Inversión Extranjera Directa (Banco Central del Paraguay).
- Autorización de los organismos competentes (permisos habilitantes conforme al sector).
- Escritura de constitución.
- Acta de la última Asamblea.
- Título de propiedad del inmueble coincidente con la licencia ambiental.
- Contrato o constancia de la entidad financiera que proveerá el crédito (si solicita el inciso "f" del Art. 5º de la Ley Nº 60/90).
- Facturas, proformas o despacho de importación (si solicita el inciso "c" del Art. 5º de la Ley Nº 60/90).
- Constancia en el Registro de Personas Jurídicas y de Beneficiarios Finales (Abogacía del Tesoro).
- Antecedentes judiciales de directores y cédula de identidad del representante legal.
- Licencia ambiental por la actividad del proyecto.
- Autorización de la SET (formato DDI).

PROCESO: La Dirección del Viceministerio de Industria realiza el chequeo documental y eleva el expediente al Consejo de Inversiones (órgano mixto público-privado), que dictamina favorablemente o solicita información complementaria. En caso de dictamen favorable, se eleva al Equipo Económico Nacional; si este aprueba el proyecto, se instrumentaliza a través de una Resolución biministerial suscrita por el Ministro de Industria y Comercio (MIC) y el Ministro de Hacienda (MH). El contrato es suscrito entre la empresa beneficiaria y el Ministro de Industria y Comercio en representación del Estado paraguayo. El organismo de aplicación es el MIC, y el MH está a cargo de los aspectos tributarios.

MARCO LEGAL: Ley Nº 5542/2015; Decreto reglamentario Nº 6100/2016.""",
    },

    # -------------------------------------------------------------------------
    # La ficha MÁS LIMPIA: norma vigente y sus reglamentos ya están en el grafo.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_eas_6480_20",
        "titulo": "Empresas por Acciones Simplificadas - EAS (Ley 6480/2020)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("Montos en guaraníes sujetos a variación (atados a los umbrales MIPYME "
                 "del Decreto 11.453/13). Verificar el monto vigente antes de citarlo."),
        "describe": ["6480", "3998/20", "DGPEJBF 02/22"],
        "texto": """OBJETO: La Ley 6480/2020 crea la Empresa por Acciones Simplificadas (EAS), una nueva PERSONERÍA JURÍDICA diseñada con un enfoque orientado a los emprendedores; un nuevo tipo societario que permite realizar una actividad lucrativa lícita en forma organizada, participando y asumiendo tanto los beneficios como las pérdidas resultantes de esta unidad económica. La EAS es una empresa de capital cuya naturaleza será siempre comercial, con independencia de las actividades previstas en su objeto social. La apertura de una EAS es totalmente EN LÍNEA a través de la web www.eas.mic.gov.py, y se accede únicamente mediante la identidad electrónica del Representante Legal principal de la empresa, de nacionalidad paraguaya o que cuente con cédula paraguaya.

PRINCIPALES BENEFICIOS Y CARACTERÍSTICAS:
- Se tramita totalmente en línea.
- Se constituye en un máximo de 72 horas y con costo CERO, utilizando los estatutos estándar.
- No establece capital mínimo ni máximo para conformarse.
- Establece la separación entre la persona física y la persona jurídica, para que el patrimonio personal del socio (o los socios) permanezca protegido. Los integrantes de la EAS responden hasta el límite de sus aportes comprometidos.
- Permite que las empresas permanezcan y crezcan, generando más empleos.
- Tributa como persona jurídica, de acuerdo con el sector de actividad e ingreso.
- Deben emitir solamente acciones nominales.
- No necesita publicar su creación, convocatorias de asamblea, disolución, etc. en un medio masivo de comunicación, ya que se publica en la página www.eas.mic.gov.py.
- La constitución podrá realizarse por contrato o acto unilateral, por medio de instrumento público o privado con certificación de firmas.
- Adquiere personalidad jurídica (distinta a la de sus integrantes) desde el momento de su inscripción en el Ministerio de Hacienda.
- No se requiere que sea inscrita en el Registro Público de Comercio para poder operar.
- Su inscripción se tramita en el Sistema Unificado de Apertura y Cierre de Empresas (SUACE), mediante un formulario único y un modelo de estatutos sociales.

REQUISITOS:
1. Identidad electrónica del Representante Legal de la empresa.
2. Representante Legal y demás autoridades de la empresa: cédula de identidad paraguaya.
3. Socios: cédula paraguaya, pasaporte, carnet de Radicación Permanente o documento de identidad del país de origen.
4. DOCUMENTOS QUE RESPALDAN EL CAPITAL INTEGRADO, según lo integrado:
   - EFECTIVO: no requiere comprobante si el aporte en efectivo no supera el monto establecido en el Art. 4º del Decreto Nº 11.453/13 (reglamenta la Ley Nº 4.457/2012 para las Micro, Pequeñas y Medianas Empresas). Este monto está sujeto a variación.
   - BOLETA DE DEPÓSITO DE GARANTÍA DEL 20%: únicamente si el aporte en efectivo supera dicho monto. Se deposita en el Banco Nacional de Fomento, cuenta número 948150, a nombre del Ministerio de Industria y Comercio (EAS) Ley Nº 6480/20.
   - BIENES REGISTRABLES (inmuebles, rodados, etc.): escritura pública del bien a nombre del socio que lo integra, consagrándose en el acto constitutivo el valor que se atribuye a los bienes aportados y los antecedentes que justifiquen esa estimación.
   - BIENES NO REGISTRABLES (equipos, mercaderías, etc.): factura comercial del bien a integrar, o inventario de valor firmado por los socios y un contador público nacional.
   - SEMOVIENTES Y OTROS: comprobantes que justifiquen su valoración, a nombre del socio que lo integra.

LAS PERSONAS JURÍDICAS ADEMÁS DEBEN PRESENTAR:
1. Cédula de identidad vigente del presidente o representante legal que tiene uso de la firma (conforme al estatuto).
2. Escritura de constitución de la empresa (socio jurídico).
3. Cédula tributaria.
4. Acta de Directorio (decisión del directorio de constituir una empresa).
5. Datos de la inscripción en el Registro Público.
6. Transcripción de la última asamblea ordinaria.

MARCO LEGAL: Ley Nº 6480/2020 (que crea la Empresa por Acciones Simplificadas EAS); Decreto Nº 3998 del 28 de agosto de 2020; Resolución Nº 623 (reglamenta el proceso de apertura de EAS); Resolución DGPEJyBF Nº 01/2021 (proceso de apertura de una EAS en Abogacía del Tesoro); Resolución DGPEJBF Nº 02/2022 (reglamenta el proceso de apertura de EAS creadas por Ley Nº 6480/2020).""",
    },

    # -------------------------------------------------------------------------
    # VIGENTE al 11/07/2026 (tomada de la web del MIC ese día).
    # HALLAZGO: esta página ACTUAL todavía exige "los requisitos exigidos por el
    # Consejo de Inversiones de la Ley 60/90" -> una ley DEROGADA. El grafo lo marca.
    # OJO: este PPA es el de POLÍTICA AUTOMOTRIZ (Ley 4838/12). NO confundir con el
    # PPA del régimen de materias primas (Decreto 11.771/00), que es otro trámite.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_ppa_automotriz_4838",
        "titulo": "Solicitud de Aprobación de Programa de Producción Anual (PPA) — Política Automotriz Nacional (Ley 4838/12)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC - Dirección de Política Automotriz Nacional",
        "fecha": "2026-07-11",
        "nota": ("Contenido tomado de la web del MIC el 11/07/2026. La ficha remite a los "
                 "requisitos del Consejo de Inversiones de la Ley 60/90, que está DEROGADA "
                 "por la Ley 7548/25. La Resolución MIC 302/13 que cita no está en el grafo."),
        "describe": ["4838", "60/90"],
        "texto": """DATOS DEL TRÁMITE:
- COSTO: MICRO sin costo; PYMES 10 jornales; GRANDES 20 jornales.
- TIEMPO DE TRAMITACIÓN: 30 días hábiles.
- MODALIDAD: presencial y en línea.
- VALIDEZ: 1 año.
- BENEFICIARIO: industrias nacionales o extranjeras habilitadas para operar bajo la Ley 4838.
- OFICINA RESPONSABLE: Dirección de Política Automotriz Nacional (DPAN).

BENEFICIOS PARA EL USUARIO:
- 0% de arancel a la importación de bienes de capital, materias primas y componentes.
- 2% de Impuesto al Valor Agregado a la importación de bienes de capital, materias primas y componentes, excepto motocicletas.
- 2% de Impuesto al Valor Agregado a la venta, excepto motocicletas.
- 20% de margen de preferencia en procesos de licitación y adquisición por parte de los Organismos y Entidades del Estado.

REQUISITOS PARA LA APROBACIÓN DEL PPA — PRIMER CICLO DE PRODUCCIÓN:
- Nota dirigida al MIC solicitando la aprobación del PPA con los beneficios de la Ley 4.838/2012.
- Formulario de Declaración Jurada de Inversión y Mano de Obra (clasificada por dependencia).
- Detalle taxativo de inversiones realizadas y registradas en el año inmediato anterior. Los valores deberán consignarse en la moneda extranjera utilizada, indicando el tipo de cambio con relación al guaraní.
- Para el primer PPA se debe adjuntar el detalle de la inversión a realizar y la mano de obra a emplear.
- Planilla de empleados asegurados en el IPS (Instituto de Previsión Social) con comprobante del último pago, y recibo de presentación de planillas laborales (Ministerio de Trabajo y Seguridad Social).
- Planilla de integración de operaciones por cada modelo y tipo de producto.
- Cumplimiento de la incorporación de procesos de fabricación establecidos en el Art. 15 inciso b) de la Resolución MIC Nº 302/13, acompañado de los documentos legales en caso de ejecutar partes y/o servicios tercerizados.
- Tabla de resumen de puntos.
- Planilla de detalle de proveedores.
- Fotografías (cuadros, chasis, kits, materia prima según configuración de importación, partes nacionales, bienes terminados, obras) según requiera la Autoridad de Aplicación (DPAN).

REQUISITOS PARA LA APROBACIÓN DEL PPA — A PARTIR DEL SEGUNDO CICLO DE PRODUCCIÓN:
- Completar los datos requeridos por el sistema y anexar en formato PDF:
- Nota dirigida al Ministerio de Industria y Comercio solicitando la aprobación del PPA con los beneficios de la Ley Nº 4838/12.
- Formulario de Declaración Jurada de Inversión y Mano de Obra (clasificada por dependencia).
- Planilla de empleados asegurados en el IPS con comprobante del último pago, y recibo de presentación de planillas laborales.
- Planillas de integración de operaciones por cada modelo y tipo de producto.
- Cumplimiento de la incorporación de procesos de fabricación establecidos en el Art. 15 inciso b) de la Resolución MIC Nº 302/13.
- Tabla de resumen de puntos.
- Planilla con detalle de proveedores.
- Fotografías según requiera la Autoridad de Aplicación (DPAN).
- Estructura del número de VIN para los nuevos modelos a producir.

PROCEDIMIENTO PASO A PASO:
PASO 1 — Presentar una nota de solicitud dirigida al Ministro de Industria y Comercio en la mesa de entrada del Ministerio, adjuntando los requisitos. Debe incluirse el original del Proyecto de Inversión, que debe contener los requisitos exigidos por el Consejo de Inversiones de la Ley 60/90 y los documentos mencionados en el artículo 4 de la Resolución 302/2013 del MIC, adjuntando:
  a) Cronograma de producción desglosado trimestralmente, especificando el destino proyectado de la producción (mercado interno, mercado externo).
  b) Cronograma de importación de materia prima, componentes, kits, partes, piezas e insumos fabriles requeridos para el cumplimiento del cronograma de producción anual, desglosados trimestralmente.
  c) Cumplimiento de la integración del Valor Agregado Nacional en los productos finales por modelo, a través de la aplicación de los procesos productivos definidos para cada tipo de producto, conforme al plazo y la cantidad de operaciones definidos en el Anexo II "Integración de Operaciones".
  d) Plano detallado de la planta industrial, layout de maquinarias y programa de ejecución.
  e) Programa de incorporación y especificaciones de los bienes de capital para el montaje de la línea de ensamblaje, cuando corresponda.
  f) Documento de autorización de la licenciataria del producto a fabricar y comercializar.
  g) Documento que indique la cantidad de trabajadores directos a emplear y sus funciones, observando la preferencia de mano de obra nacional conforme al margen establecido en el Artículo 5 inciso c) de la Ley Nº 4838/12 "Que establece la Política Automotriz Nacional".
PASO 2 — Acceder a https://www.vue.gov.py/ con usuario y contraseña, y generar la solicitud de Programa de Producción Anual completando los datos requeridos y adjuntando los documentos ya presentados en forma física.
PASO 3 — Generar la liquidación y proceder al pago.
PASO 4 — Si la solicitud cumple con los requisitos establecidos, se procede a la autorización.

CONTACTO: Dirección de Política Automotriz Nacional. Avda. Mcal. López Nº 3333 c/ Dr. Weiss, 2º piso. Tel.: (+595 21) 6163149 / 3155 / 3098. Email: politicaautomotriz@mic.gov.py""",
    },
]

# -----------------------------------------------------------------------------
# 1) PRE-CHECK: ¿existen en el grafo las normas que las fichas dicen describir?
# -----------------------------------------------------------------------------
numeros = sorted({n for f in FICHAS for n in f["describe"]})
recs, _, _ = driver.execute_query(
    """
    UNWIND $nums AS num
    OPTIONAL MATCH (n) WHERE n.numero = num AND (n:Ley OR n:Decreto OR n:Resolucion)
    RETURN num, n IS NOT NULL AS existe,
           CASE WHEN n IS NULL THEN null ELSE coalesce(n.estado,'sin_dato') END AS estado
    ORDER BY num
    """,
    nums=numeros, database_=NEO4J_DATABASE,
)
print("PRE-CHECK de normas referenciadas por las fichas:")
faltantes = []
for r in recs:
    d = r.data()
    marca = "OK " if d["existe"] else "FALTA"
    print(f"  [{marca}] {d['num']:<15} estado={d['estado']}")
    if not d["existe"]:
        faltantes.append(d["num"])
if faltantes:
    print(f"\n  AVISO: {faltantes} no existen en el grafo -> esas aristas DESCRIBE NO se crearán.")
    print("  (Las fichas igual se cargan; solo pierden ese enlace de vigencia.)")

# -----------------------------------------------------------------------------
# 2) CREAR los nodos Ficha (idempotente) y las aristas DESCRIBE (MATCH-only)
# -----------------------------------------------------------------------------
for f in FICHAS:
    driver.execute_query(
        """
        MERGE (fi:Ficha {id: $id})
        SET fi.titulo = $titulo,
            fi.texto  = $texto,
            fi.tipo   = $tipo,
            fi.estado = $estado,
            fi.fuente = $fuente,
            fi.nota   = $nota,
            fi.fecha  = $fecha
        """,
        id=f["id"], titulo=f["titulo"], texto=f["texto"], tipo=f["tipo"],
        estado=f["estado"], fuente=f["fuente"], nota=f["nota"],
        fecha=f.get("fecha"),
        database_=NEO4J_DATABASE,
    )
    # MATCH-only: si la norma no existe, NO se crea nodo fantasma.
    driver.execute_query(
        """
        MATCH (fi:Ficha {id: $id})
        UNWIND $describe AS num
        MATCH (n) WHERE n.numero = num AND (n:Ley OR n:Decreto OR n:Resolucion)
        MERGE (fi)-[:DESCRIBE]->(n)
        """,
        id=f["id"], describe=f["describe"], database_=NEO4J_DATABASE,
    )
print(f"\n{len(FICHAS)} fichas creadas/actualizadas.")

# -----------------------------------------------------------------------------
# 3) EMBEBER las fichas (mismo modelo y mismo indice que los articulos)
# -----------------------------------------------------------------------------
recs, _, _ = driver.execute_query(
    """
    MATCH (f:Ficha)
    WHERE f.texto IS NOT NULL AND f.embedding IS NULL
    RETURN elementId(f) AS eid, f.titulo AS titulo, f.texto AS texto
    """,
    database_=NEO4J_DATABASE,
)
pendientes = [r.data() for r in recs]
print(f"Fichas pendientes de embeber: {len(pendientes)}")

if pendientes:
    preparados = []
    for row in pendientes:
        # Mismo criterio que los articulos: encabezado de contexto + texto.
        texto_final, _, recortado = recortar_a_limite(f"{row['titulo']}: {row['texto']}")
        if recortado:
            print(f"  AVISO: '{row['titulo']}' se recortó por límite de tokens.")
        preparados.append((row["eid"], texto_final))

    vectores = embeber_lote([t for _, t in preparados])
    filas = [{"eid": eid, "embedding": v} for (eid, _), v in zip(preparados, vectores)]
    escribir_vectores(driver, filas)   # setea el vector Y la etiqueta :Recuperable
    print(f"  {len(filas)} fichas embebidas y guardadas.")

# -----------------------------------------------------------------------------
# 4) VERIFICACIÓN
# -----------------------------------------------------------------------------
recs, _, _ = driver.execute_query(
    """
    MATCH (f:Ficha)
    OPTIONAL MATCH (f)-[:DESCRIBE]->(n)
    RETURN f.titulo AS ficha,
           f.estado AS estado,
           f.embedding IS NOT NULL AS embebida,
           'Recuperable' IN labels(f) AS recuperable,
           collect(n.numero + ' [' + coalesce(n.estado,'sin_dato') + ']') AS describe
    ORDER BY ficha
    """,
    database_=NEO4J_DATABASE,
)
print("\nVERIFICACIÓN:")
for r in recs:
    d = r.data()
    print(f"  {d['ficha']}")
    print(f"    estado={d['estado']} | embebida={d['embebida']} | :Recuperable={d['recuperable']}")
    print(f"    describe -> {', '.join(d['describe'])}")

PRE-CHECK de normas referenciadas por las fichas:
  [OK ] 1064/97         estado=vigente
  [OK ] 22031           estado=vigente
  [OK ] 3998/20         estado=vigente
  [OK ] 4838            estado=vigente
  [OK ] 523/95          estado=vigente
  [OK ] 5542/15         estado=vigente
  [OK ] 60/90           estado=derogada
  [OK ] 6480            estado=vigente
  [OK ] 9585            estado=vigente
  [OK ] DGPEJBF 02/22   estado=vigente

6 fichas creadas/actualizadas.
Fichas pendientes de embeber: 6
  6 fichas embebidas y guardadas.

VERIFICACIÓN:
  Empresas por Acciones Simplificadas - EAS (Ley 6480/2020)
    estado=vigente | embebida=True | :Recuperable=True
    describe -> 6480 [vigente], 3998/20 [vigente], DGPEJBF 02/22 [vigente]
  Régimen de Garantía de Inversiones (Ley 5542/2015)
    estado=vigente | embebida=True | :Recuperable=True
    describe -> 5542/15 [vigente], 60/90 [derogada]
  Régimen de Incentivos Fiscales para la Inversión de Capital (Ley 60/90)
    estado=historica |

---
# 2. Retriever GraphRAG

Busca por similitud vectorial y después **expande por el grafo** con Cypher explícito:
- Si la semilla es un **artículo** → su norma padre, vigencia, y `DEROGA`/`MODIFICA`/`REGLAMENTA`.
- Si la semilla es una **ficha** → las normas que describe, con su vigencia.

Construye `retriever` y el pipeline `rag`.

In [8]:
# =============================================================================
# PARTE B — Retriever GraphRAG (ACTUALIZADO: soporta Articulo Y Ficha)
# =============================================================================
# Novedad: el nodo semilla puede ser un :Articulo (como antes) o una :Ficha.
# Si es una Ficha, se expande por DESCRIBE y se trae la VIGENCIA de cada norma
# que describe -> el grafo mantiene HONESTAS a las fichas del MIC, que citan
# normas derogadas (ej. la 60/90).
# =============================================================================

from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY = """
// --- Rama A: la semilla es un Articulo -> su norma padre ---
OPTIONAL MATCH (norma)-[:TIENE_ARTICULO]->(node)
    WHERE norma:Ley OR norma:Decreto OR norma:Resolucion
WITH node, collect(norma)[0] AS norma

// --- Rama B: la semilla es una Ficha -> las normas que describe, con vigencia ---
OPTIONAL MATCH (node)-[:DESCRIBE]->(d)
    WHERE d:Ley OR d:Decreto OR d:Resolucion
WITH node, norma, collect(CASE WHEN d IS NULL THEN null ELSE {
    numero:        toString(d.numero),
    estado:        coalesce(d.estado, 'sin_dato'),
    derogada_por:  d.derogada_por,
    vigente_hasta: d.vigente_hasta
} END) AS descritas_raw
WITH node, norma, [x IN descritas_raw WHERE x IS NOT NULL] AS descritas

RETURN
    CASE WHEN node:Ficha THEN 'ficha' ELSE 'articulo' END AS tipo_nodo,
    node.texto  AS texto,
    node.titulo AS ficha_titulo,
    node.estado AS ficha_estado,
    node.fuente AS ficha_fuente,
    node.nota   AS ficha_nota,
    descritas   AS descritas,

    // --- campos de articulo (null si la semilla es una Ficha) ---
    node.numero     AS articulo,
    node.ley_numero AS norma_numero,
    CASE WHEN norma IS NULL THEN null
         ELSE head([l IN labels(norma) WHERE l IN ['Ley','Decreto','Resolucion']]) END AS norma_tipo,
    CASE WHEN norma IS NULL THEN null ELSE norma.nombre_completo END AS norma_titulo,
    CASE WHEN norma IS NULL THEN 'sin_dato' ELSE coalesce(norma.estado,'sin_dato') END AS estado,
    CASE WHEN norma IS NULL THEN null ELSE norma.derogada_por END AS derogada_por,
    CASE WHEN norma IS NULL THEN null ELSE norma.vigente_hasta END AS vigente_hasta,
    CASE WHEN norma IS NULL THEN null ELSE norma.fuente_url END AS fuente_url,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:DEROGA]-(x)     | toString(x.numero) ] END AS derogada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:MODIFICA]-(y)   | toString(y.numero) ] END AS modificada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:REGLAMENTA]-(dd) | toString(dd.numero) + ' — ' + coalesce(dd.nombre_completo,'') ] END AS reglamentada_por,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)-[:REGLAMENTA]->(l) | toString(l.numero) + ' — ' + coalesce(l.nombre_completo,'') ] END AS reglamenta_a
"""


def _marca_vigencia(estado, derogada_por, vigente_hasta):
    """Devuelve la marca de vigencia para mostrarle al LLM."""
    if derogada_por:
        return f"[DEROGADA por {derogada_por}]"
    if vigente_hasta:
        return f"[vigente hasta {vigente_hasta}]"
    if estado and str(estado).lower() not in ("vigente", "sin_dato"):
        return f"[estado: {estado}]"
    return ""


def formatear(record):
    # =========================== FICHA ===========================
    if record.get("tipo_nodo") == "ficha":
        titulo = record.get("ficha_titulo") or "Ficha"
        fuente = record.get("ficha_fuente") or "MIC"
        encabezado = f"[FICHA {fuente}] {titulo}"
        if record.get("ficha_estado") == "historica":
            encabezado += "  [DOCUMENTO HISTÓRICO: describe un régimen DEROGADO]"

        # Vigencia HEREDADA de las normas que la ficha describe.
        # Esto es lo que impide que una ficha oficial desactualizada
        # presente como vigente un régimen que ya no lo está.
        lineas, obsoletas = [], []
        for d in (record.get("descritas") or []):
            marca = _marca_vigencia(d.get("estado"), d.get("derogada_por"), d.get("vigente_hasta"))
            lineas.append(f"{d.get('numero')} {marca}".strip())
            if d.get("derogada_por") or d.get("vigente_hasta"):
                obsoletas.append(f"{d.get('numero')} {marca}".strip())

        pie = ""
        if lineas:
            pie += "\n\nNormas que describe esta ficha: " + " · ".join(lineas)
        if obsoletas:
            pie += ("\n⚠ ADVERTENCIA: esta ficha hace referencia a normas que YA NO ESTÁN "
                    "PLENAMENTE VIGENTES (" + " · ".join(obsoletas) + "). La ficha puede estar "
                    "desactualizada en esos puntos: no presentes esa información como vigente.")

        return RetrieverResultItem(
            content=f"{encabezado}\n{record.get('texto') or ''}{pie}",
            metadata={
                "tipo": "ficha",
                "titulo": titulo,
                "estado": record.get("ficha_estado"),
                "describe": [d.get("numero") for d in (record.get("descritas") or [])],
            },
        )

    # ========================== ARTÍCULO ==========================
    marca = _marca_vigencia(
        record.get("estado"), record.get("derogada_por"), record.get("vigente_hasta")
    )
    marca = f"  {marca}" if marca else ""
    tipo = record.get("norma_tipo") or ""
    num = record.get("norma_numero") or ""
    encabezado = f"{tipo} {num}, Artículo {record.get('articulo')}{marca}".strip()

    extras = []
    if record.get("reglamentada_por"):
        extras.append("Reglamentada por: " + "; ".join(record["reglamentada_por"]))
    if record.get("reglamenta_a"):
        extras.append("Reglamenta a: " + "; ".join(record["reglamenta_a"]))
    if record.get("modificada_por_rel"):
        extras.append("Modificada por: " + ", ".join(record["modificada_por_rel"]))
    extra_txt = ("\n" + " | ".join(extras)) if extras else ""

    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}{extra_txt}",
        metadata={
            "tipo": "articulo",
            "norma": num,
            "articulo": record.get("articulo"),
            "estado": record.get("estado"),
            "derogada_por": record.get("derogada_por"),
            "fuente_url": record.get("fuente_url"),
        },
    )


retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=formatear,
    neo4j_database=NEO4J_DATABASE,
)
rag = GraphRAG(retriever=retriever, llm=llm, prompt_template=prompt_template)
print("Retriever + pipeline GraphRAG listos (soporta Artículos y Fichas).")

Retriever + pipeline GraphRAG listos (soporta Artículos y Fichas).


### Probar solo el retriever (sin LLM, barato)

Sirve para ver qué contexto trae antes de gastar en generación.

In [9]:
# Probar SOLO el retriever (sin LLM): ver qué contexto trae. Es barato.
pregunta = "¿Qué requisitos necesito para acogerme al régimen de zonas francas?"
for i, it in enumerate(retriever.search(query_text=pregunta, top_k=TOP_K).items, 1):
    print(f"[{i}] {it.metadata}")
    print(it.content[:600])
    print("---")


[1] {'tipo': 'articulo', 'norma': '523/95', 'articulo': 1, 'estado': 'vigente', 'derogada_por': None, 'fuente_url': 'https://www.mef.gov.py/sites/default/files/2025-08/1.-%20Ley%20523-1995_0.pdf'}
Ley 523/95, Artículo 1
Las Zonas Francas son espacios del territorio nacional, localizadas y autorizadas como tales por el Poder Ejecutivo, sujetas al control fiscal, aduanero y administrativo que se establece en la presente ley en las reglamentaciones pertinentes.
---
[2] {'tipo': 'ficha', 'titulo': 'Régimen de Zonas Francas (Ley 523/95)', 'estado': 'vigente', 'describe': ['60/90', '523/95']}
[FICHA MIC] Régimen de Zonas Francas (Ley 523/95)
OBJETO: Promover la atracción de inversión productiva, diversificar las exportaciones, la generación de empleo y la transferencia de conocimiento y especialización de mano de obra paraguaya.

ALCANCE: Las Zonas Francas son espacios del territorio nacional, sujetas al control fiscal, aduanero y administrativo, en las cuales se pueden desarrollar actividad

In [15]:
# Verificar que la ADVERTENCIA de la ficha llega al contexto
res = retriever.search(query_text="beneficios de instalarse en una zona franca", top_k=TOP_K)
for it in res.items:
    if it.metadata.get("tipo") == "ficha":
        print(">>>", it.metadata["titulo"])
        print(it.content[-500:])   # el FINAL, que es donde va la advertencia
        print("=" * 60)

>>> Régimen de Zonas Francas (Ley 523/95)
es.

LEGISLACIÓN RESPALDATORIA: Ley Nº 523/1995; Decreto reglamentario Nº 15554/1996; Decreto Nº 19461/2002; Decreto Nº 21309/2003; Decreto Nº 952/2018; Decreto Nº 4611/2020; Resolución General Nº 80/2021 de la SET.

Normas que describe esta ficha: 60/90 [DEROGADA por 7548/25] · 523/95
⚠ ADVERTENCIA: esta ficha hace referencia a normas que YA NO ESTÁN PLENAMENTE VIGENTES (60/90 [DEROGADA por 7548/25]). La ficha puede estar desactualizada en esos puntos: no presentes esa información como vigente.


### Pipeline completo (retriever + gpt-4o)

In [10]:
# Pipeline GraphRAG completo (retriever + gpt-4o).
pregunta = "¿Qué beneficios ofrece el régimen de zonas francas?"
print(rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)

El régimen de Zonas Francas en Paraguay ofrece varios beneficios, entre los cuales se destacan:

1. **Importación de materia prima o mercaderías con una tasa de 0%**: Esto permite a los usuarios importar insumos sin pagar aranceles, facilitando la reducción de costos en la producción (Ley 523/95).

2. **Impuesto al Valor Agregado (IVA) 0%**: Las transacciones dentro de las Zonas Francas están exentas del IVA, lo que representa un ahorro significativo para las empresas que operan allí (Ley 523/95).

3. **Servicios y comercios entre usuarios de Zonas Francas: 0% de impuestos**: Las operaciones comerciales y de servicios entre usuarios dentro de las Zonas Francas no están sujetas a impuestos, incentivando la colaboración y el comercio interno (Ley 523/95).

4. **Exportación a terceros países: 0,5% del valor factura**: Las exportaciones realizadas desde las Zonas Francas a otros países están sujetas a un impuesto reducido del 0,5% sobre el valor de la factura (Ley 523/95).

5. **Emisión de

---
# 3. Baseline y comparación

El **baseline** es el mismo sistema con el **grafo apagado**: idéntico embedder, LLM, prompt y `top_k`, pero sin expansión por el grafo.

Como lo único que cambia es la recuperación, **cualquier diferencia en las respuestas es atribuible al grafo**. Es el experimento central de la tesis.

In [11]:
# BASELINE vector-only: MISMO embedder, LLM, prompt y top_k, pero el retrieval_query
# NO expande al grafo (solo devuelve el texto). Como lo único que cambia es la
# recuperación, cualquier diferencia en las respuestas es atribuible AL GRAFO.
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY_BASELINE = """
RETURN node.texto AS texto, node.numero AS articulo, node.ley_numero AS norma_numero
"""

def formatear_baseline(record):
    num = record.get("norma_numero") or ""
    art = record.get("articulo")
    encabezado = f"Norma {num}, Artículo {art}".strip() if num or art else "Documento"
    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}",
        metadata={"norma": num, "articulo": art},
    )

retriever_baseline = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY_BASELINE,
    embedder=embedder,
    result_formatter=formatear_baseline,
    neo4j_database=NEO4J_DATABASE,
)
rag_baseline = GraphRAG(retriever=retriever_baseline, llm=llm, prompt_template=prompt_template)
print("Baseline listo.")

Baseline listo.


### Comparar

Dos casos testigo, ambos sobre leyes **derogadas**. Lo esperable: el baseline las presenta como vigentes (la falla); el GraphRAG advierte la derogación (el acierto).

In [12]:
# Comparar baseline vs GraphRAG sobre la MISMA pregunta.
_faltan = [n for n in ("TOP_K", "rag", "rag_baseline") if n not in globals()]
if _faltan:
    print("⚠ Faltan objetos:", ", ".join(_faltan))
    print("   → Corré todas las celdas de arriba en orden.")
    print("   Atajo: Runtime → 'Run before' parado en esta celda.")
else:
    for pregunta in [
        "¿qué incentivos fiscales ofrece la Ley 60/90?",     # DEROGADA por 7548/25
        "¿qué establece la Ley 5102 sobre alianza público-privada?",  # DEROGADA por 7452/25
    ]:
        print("=" * 70)
        print("PREGUNTA:", pregunta)
        print("\n--- BASELINE (vector-only, grafo apagado) ---")
        print(rag_baseline.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)
        print("\n--- GraphRAG (con grafo) ---")
        print(rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)
        print()

PREGUNTA: ¿qué incentivos fiscales ofrece la Ley 60/90?

--- BASELINE (vector-only, grafo apagado) ---
La Ley N° 60/90 ofrece incentivos fiscales con el objetivo de promover e incrementar las inversiones de capital de origen nacional y/o extranjero. Los beneficios fiscales se otorgan a personas físicas y jurídicas radicadas en Paraguay, cuyas inversiones se alineen con la política económica y social del Gobierno Nacional. Los objetivos específicos para otorgar estos beneficios incluyen:

a) El acrecentamiento de la producción de bienes y servicios.
b) La creación de fuentes de trabajo permanente.
c) El fomento de las exportaciones y la sustitución de importaciones.
d) La incorporación de tecnologías que aumenten la eficiencia productiva y permitan una mejor utilización de materias primas, mano de obra y recursos energéticos nacionales.
e) La inversión y reinversión de utilidades en bienes de capital.

Estos incentivos están detallados en el Artículo 1 de la Ley N° 60/90.

--- GraphRAG 

---
# 4. Agente conversacional (LangGraph)

Flujo: **reescribir → recuperar → graduar → { responder | reformular↻ | sin_contexto }**

- **Memoria**: recuerda la conversación (mismo `thread_id`).
- **Reescritura**: convierte "¿y qué actividades permite?" en una consulta autónoma antes de buscar.
- **Corrective RAG**: si el contexto no sirve, reformula y vuelve a buscar; si sigue sin servir, admite que no sabe en vez de inventar.

In [13]:
# AGENTE CONVERSACIONAL (LangGraph)
# Grafo: reescribir -> recuperar -> graduar -> { responder | reformular↻ | sin_contexto }
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si una FICHA advierte que referencia normas no vigentes, "
    "trasladá esa advertencia. Si el contexto no alcanza, decilo."
)

class EstadoConv(TypedDict):
    messages: Annotated[list, operator.add]
    consulta: str      # consulta (reescrita) con la que se busca
    context: str       # contexto recuperado
    intentos: int      # cuántas búsquedas se hicieron (corta el bucle)
    relevante: bool    # veredicto del grading

def reescribir(state: EstadoConv):
    # Convierte una pregunta de seguimiento en una consulta AUTÓNOMA usando el historial,
    # para que la RECUPERACIÓN también aproveche el contexto de la conversación.
    if len(state["messages"]) <= 1:
        return {"consulta": state["messages"][-1]["content"], "intentos": 0}
    historial = "\n".join(f'{m["role"]}: {m["content"]}' for m in state["messages"][:-1])
    pregunta = state["messages"][-1]["content"]
    prompt = (
        "Dada la conversación previa y una pregunta de seguimiento, reescribí la pregunta como "
        "una consulta AUTÓNOMA que se entienda sin el historial (resolvé 'esa ley', 'ese régimen'). "
        "Si ya es autónoma, devolvela igual. Devolvé SOLO la consulta, sin comillas.\n\n"
        f"# Conversación:\n{historial}\n\n# Pregunta:\n{pregunta}\n\n# Consulta autónoma:"
    )
    return {"consulta": llm.invoke(prompt).content.strip(), "intentos": 0}

def recuperar(state: EstadoConv):
    res = retriever.search(query_text=state["consulta"], top_k=TOP_K)
    return {"context": "\n\n".join(it.content for it in res.items),
            "intentos": state.get("intentos", 0) + 1}

def graduar(state: EstadoConv):
    # OJO: sin la aclaración de abajo, el grader lee "norma DEROGADA" como
    # "contexto insuficiente" y manda al fallback, TAPANDO la advertencia de vigencia
    # (que es justo el aporte del sistema). Bug real detectado con la Ley 5102.
    prompt = (
        "Decidí si el CONTEXTO alcanza para responder la PREGUNTA de forma fundamentada. "
        "Respondé SOLO con 'SI' o 'NO'.\n\n"
        "IMPORTANTE: si el contexto contiene artículos o fichas sobre la norma consultada, "
        "alcanza — AUNQUE la norma figure como DEROGADA, modificada o no vigente. Advertir que "
        "una norma está derogada (y por cuál fue reemplazada) es una respuesta VÁLIDA y útil, "
        "no una falta de información. Respondé 'NO' solo si el contexto es de otra materia.\n\n"
        f"# Pregunta:\n{state['consulta']}\n\n# Contexto:\n{state['context'][:4000]}\n\n# ¿Alcanza? (SI/NO):"
    )
    return {"relevante": llm.invoke(prompt).content.strip().upper().startswith("SI")}

def decidir(state: EstadoConv):
    # Router: SOLO lee el estado y devuelve el nombre del próximo nodo.
    if state["relevante"]:
        return "responder"
    if state["intentos"] < 2:
        return "reformular"
    return "sin_contexto"

def reformular(state: EstadoConv):
    prompt = (
        "La búsqueda anterior no trajo contexto suficiente. Reformulá la consulta con otros "
        "términos o de forma más general. Devolvé SOLO la nueva consulta.\n\n"
        f"# Consulta anterior:\n{state['consulta']}"
    )
    return {"consulta": llm.invoke(prompt).content.strip()}

def responder(state: EstadoConv):
    resp = llm.invoke(
        input=f"# Contexto:\n{state['context']}\n\n# Pregunta:\n{state['consulta']}",
        message_history=state["messages"][:-1],
        system_instruction=SYSTEM,
    )
    return {"messages": [{"role": "assistant", "content": resp.content}]}

def sin_contexto(state: EstadoConv):
    return {"messages": [{"role": "assistant", "content":
        "No encontré en la base normativa cargada información suficiente para responder eso "
        "con fundamento. ¿Podés reformular la pregunta o dar más detalle?"}]}

builder = StateGraph(EstadoConv)
for nombre, fn in [("reescribir", reescribir), ("recuperar", recuperar), ("graduar", graduar),
                   ("reformular", reformular), ("responder", responder), ("sin_contexto", sin_contexto)]:
    builder.add_node(nombre, fn)
builder.add_edge(START, "reescribir")
builder.add_edge("reescribir", "recuperar")
builder.add_edge("recuperar", "graduar")
builder.add_conditional_edges("graduar", decidir, {
    "responder": "responder", "reformular": "reformular", "sin_contexto": "sin_contexto",
})
builder.add_edge("reformular", "recuperar")   # bucle acotado por intentos < 2
builder.add_edge("responder", END)
builder.add_edge("sin_contexto", END)

chat_graph = builder.compile(checkpointer=InMemorySaver())
print("Agente conversacional listo.")

Agente conversacional listo.


### Demo — las cuatro rutas

Memoria+reescritura · ficha operativa · norma derogada · fuera de dominio.

In [16]:
import uuid

def preguntar(chat_id, texto):
    out = chat_graph.invoke({"messages": [{"role": "user", "content": texto}]},
                            {"configurable": {"thread_id": chat_id}})
    print("Usuario:", texto)
    print("  (consulta usada:", out["consulta"], "| búsquedas:", out["intentos"], ")")
    print("Bot:", out["messages"][-1]["content"], "\n")

# 1) Memoria + reescritura: el 2do turno no nombra la ley
chat = str(uuid.uuid4())
preguntar(chat, "¿Qué es el régimen de zonas francas?")
preguntar(chat, "¿Qué actividades permite?")

# 2) FICHA: requisitos operativos (no están en el articulado)
preguntar(str(uuid.uuid4()), "¿Cómo abro una Empresa por Acciones Simplificadas (EAS)?")

# 3) NORMA DEROGADA: debe ADVERTIR, no decir "no sé"
preguntar(str(uuid.uuid4()), "¿qué establece la Ley 5102 sobre alianza público-privada?")

# 4) FUERA DE DOMINIO: fallback honesto
preguntar(str(uuid.uuid4()), "¿Cuál es la mejor receta de sopa paraguaya?")

# Preguntas que apuntan al punto obsoleto de las fichas
preguntar(str(uuid.uuid4()), "¿Qué incentivos fiscales obtengo si me instalo en una zona franca?")
preguntar(str(uuid.uuid4()), "¿Qué requisitos necesito para el PPA de la política automotriz?")

Usuario: ¿Qué es el régimen de zonas francas?
  (consulta usada: ¿Qué es el régimen de zonas francas? | búsquedas: 1 )
Bot: El régimen de Zonas Francas en Paraguay, según la Ley 523/95, se refiere a espacios del territorio nacional que son localizados y autorizados por el Poder Ejecutivo. Estas zonas están sujetas a control fiscal, aduanero y administrativo, y se pueden desarrollar en ellas actividades comerciales, industriales y de servicios (Ley 523/95, Artículo 1). Las Zonas Francas deben instalarse en áreas de propiedad privada, cercadas para garantizar su aislamiento respecto del Territorio Aduanero, y no constituyen parte del Territorio Aduanero de Paraguay (Ley 523/95, Artículo 2).

El objetivo de este régimen es promover la atracción de inversión productiva, diversificar las exportaciones, generar empleo y facilitar la transferencia de conocimiento y especialización de la mano de obra paraguaya. Los concesionarios son personas jurídicas que, mediante contrato con el Poder Ejecu

---
## Notas

- **`TOP_K`** = cuántos documentos recupera (no es la ventana de contexto). Más alto = más contexto y más costo.
- **Números de norma** (`60/90`) se buscan con **match exacto**, no con fulltext (el tokenizer parte el `/`).
- **La base NO se llama `neo4j`**: es `ece63d51`. Viene del secreto `NEO4J_DATABASE`.
- **Pendiente**: cargar el articulado de la **Ley 7452/25** (deroga la 5102). Hasta entonces, el sistema avisa que la 5102 está derogada pero no puede detallar el régimen nuevo. Su ficha de PPP queda sin cargar por eso.
- **Fuera de alcance por ahora**: reglamentos citados por las fichas que no están en el grafo (Decreto 15554/96, 6100/2016, Resolución MIC 302/13, etc.). Se cargarán solo si alguna respuesta falla por su ausencia.